In [28]:
#ADD THE DATASET ROWS IN THE Datasets and ApplicationDomain SHEET OF Fairness-Metrics.xlsx 
# TO THE ONTOLOGY papers.ttl, WITH DESCRIPTION, LINK AND RELATIONS

from rdflib import Graph
import pandas as pd
from rdflib import URIRef,Literal, Namespace 
from rdflib.namespace import DC,RDF,RDFS,split_uri

In [ ]:
# Creating a copy of a given ttl file
g = Graph()
g.parse("./complete-ontologies/infiles/input-papers.ttl", format='turtle')
print(len(g))


39863


In [30]:

# Namespaces
CORE = Namespace("http://fairness-ontology.com/core#")
PAPERS = Namespace("http://fairness-ontology.com/papers#")
g.bind("core", CORE)
g.bind("dc", DC)
g.bind("rdfs", RDFS)
g.bind("papers", PAPERS)

# WE add a connection with the core: ontology:
##BIBTEX = Namespace(str(dict(g.namespaces()).get("bibtex")))
#BIBTEX = Namespace(str(dict(g.namespaces()).get("j.1")))
#g.add((BIBTEX.Article, RDFS.subClassOf, CORE.ScientificPaper))
#g.add((BIBTEX.InBook, RDFS.subClassOf, CORE.ScientificPaper))
#g.add((BIBTEX.InCollection, RDFS.subClassOf, CORE.ScientificPaper))
#g.add((BIBTEX.InProceedings, RDFS.subClassOf, CORE.ScientificPaper))
#g.add((BIBTEX.Misc, RDFS.subClassOf, CORE.ScientificPaper))
#g.add((BIBTEX.PhdThesis, RDFS.subClassOf, CORE.ScientificPaper))
#g.add((BIBTEX.TechReport, RDFS.subClassOf, CORE.ScientificPaper))


In [27]:

# load the content of the Datasets and ApplicationDomain excel sheet

df = pd.read_excel("../Fairness-Metrics.xlsx", sheet_name="Datasets and ApplicationDomain",header=0, dtype=str)
df.fillna('', inplace=True)


def camelCaseString(s):  
    temp = s.replace('_', ' ').replace('-', ' ').split('(')[0]
    temp = ' '.join([w.title() if w.islower() else w for w in temp.split()])
    temp=temp.replace(' ', '')
    #res = temp[0].lower() + temp[1:]
    return temp

# Iterate over rows
for _, row in df.iterrows():
    dataset_name = camelCaseString(str(row["Dataset"]).strip())
    datasetURI = PAPERS[dataset_name] #URIRef(CORE["papers/" + dataset_name])
    
    # Dataset type
    g.add((datasetURI, RDF.type, CORE.Dataset))
    
    # Description as rdfs:comment
    if pd.notna(row["Description"]):
        g.add((datasetURI, RDFS.comment, Literal(str(row["Description"]).strip())))
    
    # Link as dc:source
    if pd.notna(row["link"]):
        g.add((datasetURI, DC.source, Literal(str(row["link"]).strip())))
    
    # Application domain
    if pd.notna(row["ApplicationDomain"]):
        domains = [d.strip() for d in str(row["ApplicationDomain"]).split(",")]
        for domain in domains:
            if domain:
                domainURI = URIRef(CORE[domain])
                
                #g.add((domainURI, RDF.type, CORE.ApplicationDomain))
                g.add((datasetURI, CORE.relatesTo, domainURI))
   
    # Scientific papers (comma-separated)
    if pd.notna(row["ScientificPaper"]):
        papers = [p.strip().replace("-","_") for p in str(row["ScientificPaper"]).split(",")]
        
        for paper in papers:
            if paper:  # avoid empty strings
                #paperURI = URIRef(FAIR["papers/" + paper])
                if (PAPERS[paper], RDF.type, None) not in g:
                    print("The ontology doesn not contain "+paper+" yet. ")
                #g.add((PAPERS[paper], RDF.type, CORE.ScientificPaper))
                g.add((PAPERS[paper], CORE.testedOn, datasetURI))

# Save graph
g.serialize("../outfiles/papers_upd2.ttl", format="turtle")
print("Ontology saved to papers_upd2.ttl")

Ontology saved to papers_upd2.ttl
